<a href="https://colab.research.google.com/github/anajrubim/Atividade_IHC/blob/main/ollamo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import sys

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
  print('Running on Colab. Installing dependencies...')
  !sudo apt-get update && sudo apt-get install -y zstd
  print('Dependencies installed.')
else:
  print('Not running on Colab. Please install zstd manually if needed.')

Running on Colab. Installing dependencies...

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,000 kB]
Fetched 13.2 MB in 2s (5,513 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources

Dependencies installed.

In [ ]:
import threading
import subprocess

# Start Ollama service in background

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

In [ ]:
!ollama pull granite4:350m

In [ ]:
### https://docs.ollama.com/capabilities/tool-calling#python
###https://ollama.com/library/granite4

In [ ]:
!uv pip install ollama

Using Python 3.12.13 environment at: /usr
Checked 1 package in 104ms


In [ ]:
!uv pip install rich

Using Python 3.12.13 environment at: /usr
Checked 1 package in 93ms


In [ ]:
import json
from rich import print
from ollama import chat
model = 'granite4:350m'

In [ ]:
 import requests

url = "https://store.steampowered.com/api/featuredcategories"

resposta = requests.get(url)

print(resposta)


<Response [200]>

In [ ]:
import requests
import sqlite3

url = "https://store.steampowered.com/api/featuredcategories/"

resposta = requests.get(url)
dados = resposta.json()

conn = sqlite3.connect("steam.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS jogos (
    id INTEGER PRIMARY KEY,
    nome TEXT,
    preco_original INTEGER,
    preco_final INTEGER,
    desconto INTEGER,
    categoria TEXT
)
""")

for categoria, bloco in dados.items():

    if not isinstance(bloco, dict):
        continue

    items = bloco.get("items")

    if not items:
        continue

    for jogo in items:

        if "id" not in jogo:
            continue

        id_jogo = jogo.get("id")
        nome = jogo.get("name")

        preco_original = jogo.get("original_price")
        preco_final = jogo.get("final_price")
        desconto = jogo.get("discount_percent")

        if preco_original is None or preco_final is None:
            price = jogo.get("price_overview", {})

            preco_original = price.get("initial")
            preco_final = price.get("final")
            desconto = price.get("discount_percent")

        cursor.execute("""
        INSERT OR IGNORE INTO jogos
        (id, nome, preco_original, preco_final, desconto, categoria)
        VALUES (?, ?, ?, ?, ?, ?)
        """, (
            id_jogo,
            nome,
            preco_original,
            preco_final,
            desconto,
            categoria
        ))

conn.commit()
conn.close()


In [ ]:

! pip install pandas
import pandas as pd


In [ ]:
def carregar_dados():
    conn = sqlite3.connect("steam.db")
    df = pd.read_sql("SELECT * FROM jogos", conn)
    conn.close()
    return df


In [ ]:
def tratar_precos(df):
    df["preco_original"] = df["preco_original"] / 100
    df["preco_final"] = df["preco_final"] / 100
    return df

In [ ]:
def maiores_descontos(n=10, **kwargs):

    df = carregar_dados()
    df = tratar_precos(df)

    return df.sort_values("desconto", ascending=False).head(n)



In [ ]:
def jogos_baratos(preco=5, **kwargs):

    df = carregar_dados()
    df = tratar_precos(df)

    return df[df["preco_final"] < preco]




In [ ]:
def media_por_categoria(**kwargs):

    df = carregar_dados()
    df = tratar_precos(df)

    return df.groupby("categoria")["preco_final"].mean()

In [ ]:
def jogos_gratis(**kwargs):

    df = carregar_dados()
    df = tratar_precos(df)

    return df[df["preco_final"] == 0]

In [ ]:
def jogos_mais_caros(n=10, **kwargs):

    df = carregar_dados()
    df = tratar_precos(df)

    return df.sort_values(
        "preco_final",
        ascending=False
    ).head(n)

In [ ]:
def busca_jogo(df, jogo):
    resultado = df[df["nome"].str.contains(jogo, case=False, na=False)]

    if resultado.empty:
        return "Nenhum jogo encontrado."

    return resultado[["id", "nome", "preco_original", "preco_final", "desconto", "categoria"]]

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import threading
import subprocess
import time
import os

def run_ollama_serve():
  try:
    # Use preexec_fn=os.setsid to detach the process from the current shell
    subprocess.Popen(["ollama", "serve"], preexec_fn=os.setsid, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
  except Exception as e:
    print(f"Error starting Ollama service: {e}")

thread = threading.Thread(target=run_ollama_serve)
thread.start()

# Give Ollama a moment to start up
time.sleep(15) # Increased sleep time for robustness
print("Ollama service started in background (hopefully).")


Ollama service started in background (hopefully).

In [ ]:
import time

time.sleep(5) # Give the service a little more time before pulling
!ollama list # Check if ollama command is recognized
!ollama pull granite4:350m

NAME             ID              SIZE      MODIFIED           
granite4:350m    5eee845b49c4    708 MB    About a minute ago    



In [ ]:
from rich import print
from ollama import chat

model = 'granite4:350m'
messages = [{'role': 'user', 'content': 'Explique o conceito de inteligência artificial em termos simples.'}]

print('Enviando mensagem para o modelo Ollama...')
response = chat(model, messages=messages)
print('Resposta do Modelo:', response['message']['content'])

Enviando mensagem para o modelo Ollama...

Resposta do Modelo: Inteligência Artificial (IA) é um campo da ciência humana que se concentra na criação de 
sistemas capazes de realizar tarefas que exige inteligência, como entender e aprender com informações, responder a 
perguntas, realizar tarefas específicas, etc.

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"], preexec_fn=subprocess.os.setsid, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama_serve)
thread.start()

time.sleep(5)
print("Ollama service started in background (hopefully).")

Ollama service started in background (hopefully).

In [ ]:
pergunta = input("O que você quer procurar na Steam? ")

messages = [{'role': 'user', 'content': pergunta}]

df = carregar_dados()
df = tratar_precos(df)

tools_map = {
    "carregar_dados": carregar_dados,
    "tratar_precos": tratar_precos,
    "maiores_descontos": maiores_descontos,
    "jogos_baratos": jogos_baratos,
    "media_por_categoria": media_por_categoria,
    "busca_jogo":busca_jogo,
    "jogos_gratis": jogos_gratis,
    "jogos_mais_caros": jogos_mais_caros
}

response = chat(
    model=model,
    messages=messages,
    tools=list(tools_map.values())
)

if response.message.tool_calls:

    tool = response.message.tool_calls[0]

    nome_funcao = tool.function.name
    argumentos = tool.function.arguments or {}

    func = tools_map[nome_funcao]

    argumentos_filtrados = {}

    for k, v in argumentos.items():
        if k != "df" and k in func.__code__.co_varnames:
            argumentos_filtrados[k] = v

    if "df" in func.__code__.co_varnames:
        argumentos_filtrados["df"] = df

    result = func(**argumentos_filtrados)

    print(result)

else:
    print(response.message.content)


O que você quer procurar na Steam? jogos mais caros


id                          nome  preco_original  preco_final  \
4   2483190               Forza Horizon 6           69.99        69.99   
1   1086940               Baldur's Gate 3           59.99        44.99   
6   3041230                      Windrose           29.99        29.99   
7   3110760                Alabaster Dawn           24.99        21.24   
9   3404260                 Dead as Disco           24.99        19.99   
0    311210  Call of Duty®: Black Ops III           59.99        19.79   
5   2582320                       Mixtape           19.99        17.99   
8   3124540                  Far Far West           19.99        17.99   
12  3672400                       Farever           19.99        17.99   
25  4337790                    Akuma Rise           19.99        17.99   

    desconto     categoria  
4        0.0   top_sellers  
1       25.0      specials  
6        0.0   top_sellers  
7       15.0      specials  
9       20.0      specials  
0       67.0      specials  
5       10.0      specials  
8       10.0      specials  
12      10.0      specials  
25      10.0  new_releases

In [ ]:
from ollama import chat

print('Demonstrating a tool call for "jogos baratos" (cheap games)...')

model = 'granite4:350m'

demo_message = [{'role': 'user', 'content': 'Quero ver os jogos mais baratos.'}]

demo_response = chat(
    model=model,
    messages=demo_message,
    tools=list(tools_map.values())
)

if demo_response.message.tool_calls:
    tool = demo_response.message.tool_calls[0]

    nome_funcao = tool.function.name
    argumentos = tool.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao}")
    print(f"Arguments: {argumentos}")

    func = tools_map[nome_funcao]

    argumentos_filtrados = {}

    for k, v in argumentos.items():
        if k != "df" and k in func.__code__.co_varnames:
            argumentos_filtrados[k] = v

    df_current = carregar_dados()
    df_current = tratar_precos(df_current)

    if "df" in func.__code__.co_varnames:
        argumentos_filtrados["df"] = df_current

    result = func(**argumentos_filtrados)

    print("\nTool execution result:")
    print(result)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response.message.content)


Demonstrating a tool call for "jogos baratos" (cheap games)...

Ollama suggested tool: jogos_baratos

Arguments: {'kwargs': {}}

Tool execution result:

id                                   nome  preco_original  \
16  3892270               Gamble With Your Friends            7.99   
17  4042600                         Arcane Plunder            3.99   
19  4171250                            Cyber Waifu            1.99   
21  4240840                            Intersectio            3.99   
27  4440830  Hentai Tales: Licentious Of Chinatown            0.99   
32  4532900                        Haunted Izakaya            4.99   
35  4540950    Everything is Crab - Supporter Pack            4.99   
41  4592100                            Umigami Eye            2.99   

    preco_final  desconto     categoria  
16         4.95      38.0      specials  
17         2.99      25.0  new_releases  
19         1.19      40.0  new_releases  
21         3.99       0.0  new_releases  
27         0.59      40.0  new_releases  
32         4.49      10.0  new_releases  
35         4.99       0.0  new_releases  
41         2.69      10.0  new_releases

In [ ]:
print('Demonstrating a tool call for "maiores descontos" (top discounts)...')

demo_message_discounts = [{'role': 'user', 'content': 'Quero ver os jogos com os maiores descontos.'}]

demo_response_discounts = chat(
    model=model,
    messages=demo_message_discounts,
    tools=list(tools_map.values())
)

if demo_response_discounts.message.tool_calls:
    tool_discounts = demo_response_discounts.message.tool_calls[0]

    nome_funcao_discounts = tool_discounts.function.name
    argumentos_discounts = tool_discounts.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao_discounts}")
    print(f"Arguments: {argumentos_discounts}")

    func_discounts = tools_map[nome_funcao_discounts]

    argumentos_filtrados_discounts = {}

    for k, v in argumentos_discounts.items():
        if k != "df" and k in func_discounts.__code__.co_varnames:
            argumentos_filtrados_discounts[k] = v

    df_current_discounts = carregar_dados()
    df_current_discounts = tratar_precos(df_current_discounts)

    if "df" in func_discounts.__code__.co_varnames:
        argumentos_filtrados_discounts["df"] = df_current_discounts

    result_discounts = func_discounts(**argumentos_filtrados_discounts)

    print("\nTool execution result for top discounts:")
    print(result_discounts)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response_discounts.message.content)


Demonstrating a tool call for "maiores descontos" (top discounts)...

Ollama suggested tool: maiores_descontos

Arguments: {'kwargs': {}}

Tool execution result for top discounts:

id                                      nome  preco_original  \
0    311210              Call of Duty®: Black Ops III           59.99   
3   1659420  UNCHARTED™: Legacy of Thieves Collection           49.99   
2   1634150      Behind the Frame: The Finest Scenery           12.99   
31  4522660                         Futanari Mermaids            9.99   
27  4440830     Hentai Tales: Licentious Of Chinatown            0.99   
19  4171250                               Cyber Waifu            1.99   
16  3892270                  Gamble With Your Friends            7.99   
1   1086940                           Baldur's Gate 3           59.99   
17  4042600                            Arcane Plunder            3.99   
14  3864610                 Kodoku：As the Moon Mourns            7.90   

    preco_final  desconto     categoria  
0         19.79      67.0      specials  
3         16.49      67.0      specials  
2          6.49      50.0             6  
31         5.99      40.0  new_releases  
27         0.59      40.0  new_releases  
19         1.19      40.0  new_releases  
16         4.95      38.0      specials  
1         44.99      25.0      specials  
17         2.99      25.0  new_releases  
14         6.32      20.0  new_releases

In [ ]:
print('Demonstrating a tool call for "media por categoria" (average price by category)...')

demo_message_avg_price = [{'role': 'user', 'content': 'Qual é a média de preço dos jogos por categoria?'}]

demo_response_avg_price = chat(
    model=model,
    messages=demo_message_avg_price,
    tools=list(tools_map.values())
)

if demo_response_avg_price.message.tool_calls:
    tool_avg_price = demo_response_avg_price.message.tool_calls[0]

    nome_funcao_avg_price = tool_avg_price.function.name
    argumentos_avg_price = tool_avg_price.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao_avg_price}")
    print(f"Arguments: {argumentos_avg_price}")

    func_avg_price = tools_map[nome_funcao_avg_price]

    argumentos_filtrados_avg_price = {}

    for k, v in argumentos_avg_price.items():
        if k != "df" and k in func_avg_price.__code__.co_varnames:
            argumentos_filtrados_avg_price[k] = v

    df_current_avg_price = carregar_dados()
    df_current_avg_price = tratar_precos(df_current_avg_price)

    if "df" in func_avg_price.__code__.co_varnames:
        argumentos_filtrados_avg_price["df"] = df_current_avg_price

    result_avg_price = func_avg_price(**argumentos_filtrados_avg_price)

    print("\nTool execution result for average price by category:")
    print(result_avg_price)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response_avg_price.message.content)


Demonstrating a tool call for "media por categoria" (average price by category)...

Ollama suggested tool: media_por_categoria

Arguments: {'kwargs': ''}

Tool execution result for average price by category:

categoria
6                6.490000
coming_soon           NaN
new_releases     6.912778
specials        19.041000
top_sellers     49.990000
Name: preco_final, dtype: float64

In [ ]:
print('Demonstrating a tool call for "busca jogo" (search game)...')

demo_message_search_game = [{'role': 'user', 'content': 'Procure pelo jogo "The Greenening"'}]

demo_response_search_game = chat(
    model=model,
    messages=demo_message_search_game,
    tools=list(tools_map.values())
)

if demo_response_search_game.message.tool_calls:
    tool_search_game = demo_response_search_game.message.tool_calls[0]

    nome_funcao_search_game = tool_search_game.function.name
    argumentos_search_game = tool_search_game.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao_search_game}")
    print(f"Arguments: {argumentos_search_game}")

    func_search_game = tools_map[nome_funcao_search_game]

    argumentos_filtrados_search_game = {}

    df_current_search_game = carregar_dados()
    df_current_search_game = tratar_precos(df_current_search_game)

    if "df" in func_search_game.__code__.co_varnames:
        argumentos_filtrados_search_game["df"] = df_current_search_game

    if 'jogo' in argumentos_search_game:
        argumentos_filtrados_search_game['jogo'] = argumentos_search_game['jogo']

    result_search_game = func_search_game(**argumentos_filtrados_search_game)

    print("\nTool execution result for game search:")
    print(result_search_game)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response_search_game.message.content)


Demonstrating a tool call for "busca jogo" (search game)...

Ollama suggested tool: busca_jogo

Arguments: {'jogo': 'The Greenening'}

Tool execution result for game search:

Nenhum jogo encontrado.

In [ ]:
from ollama import chat
model = 'granite4:350m'

print('Demonstrating a tool call for "jogos_gratis" (free games)...')

demo_message_free_games = [{'role': 'user', 'content': 'Quais são os jogos grátis?'}]

demo_response_free_games = chat(
    model=model,
    messages=demo_message_free_games,
    tools=list(tools_map.values())
)

if demo_response_free_games.message.tool_calls:
    tool_free_games = demo_response_free_games.message.tool_calls[0]

    nome_funcao_free_games = tool_free_games.function.name
    argumentos_free_games = tool_free_games.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao_free_games}")
    print(f"Arguments: {argumentos_free_games}")

    func_free_games = tools_map[nome_funcao_free_games]

    argumentos_filtrados_free_games = {}

    for k, v in argumentos_free_games.items():
        if k != "df" and k in func_free_games.__code__.co_varnames:
            argumentos_filtrados_free_games[k] = v

    df_current_free_games = carregar_dados()
    df_current_free_games = tratar_precos(df_current_free_games)

    if "df" in func_free_games.__code__.co_varnames:
        argumentos_filtrados_free_games["df"] = df_current_free_games

    result_free_games = func_free_games(**argumentos_filtrados_free_games)

    print("\nTool execution result for free games:")
    print(result_free_games)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response_free_games.message.content)


Demonstrating a tool call for "jogos_gratis" (free games)...

Ollama suggested tool: jogos_gratis

Arguments: {'kwargs': {}}

Tool execution result for free games:

Empty DataFrame
Columns: [id, nome, preco_original, preco_final, desconto, categoria]
Index: []

In [ ]:
from ollama import chat
model = 'granite4:350m'

print('Demonstrating a tool call for "jogos_mais_caros" (most expensive games)...')

demo_message_expensive_games = [{'role': 'user', 'content': 'Quais são os 5 jogos mais caros?'}]

demo_response_expensive_games = chat(
    model=model,
    messages=demo_message_expensive_games,
    tools=list(tools_map.values())
)

if demo_response_expensive_games.message.tool_calls:
    tool_expensive_games = demo_response_expensive_games.message.tool_calls[0]

    nome_funcao_expensive_games = tool_expensive_games.function.name
    argumentos_expensive_games = tool_expensive_games.function.arguments or {}

    print(f"Ollama suggested tool: {nome_funcao_expensive_games}")
    print(f"Arguments: {argumentos_expensive_games}")

    func_expensive_games = tools_map[nome_funcao_expensive_games]

    argumentos_filtrados_expensive_games = {}

    for k, v in argumentos_expensive_games.items():
        if k != "df" and k in func_expensive_games.__code__.co_varnames:
            argumentos_filtrados_expensive_games[k] = v

    df_current_expensive_games = carregar_dados()
    df_current_expensive_games = tratar_precos(df_current_expensive_games)

    if "df" in func_expensive_games.__code__.co_varnames:
        argumentos_filtrados_expensive_games["df"] = df_current_expensive_games

    result_expensive_games = func_expensive_games(**argumentos_filtrados_expensive_games)

    print("\nTool execution result for most expensive games:")
    print(result_expensive_games)

else:
    print("\nOllama did not suggest a tool. Model response:")
    print(demo_response_expensive_games.message.content)


Demonstrating a tool call for "jogos_mais_caros" (most expensive games)...

Ollama suggested tool: jogos_mais_caros

Arguments: {'kwargs': 'n'}

Tool execution result for most expensive games:

id                          nome  preco_original  preco_final  \
4   2483190               Forza Horizon 6           69.99        69.99   
1   1086940               Baldur's Gate 3           59.99        44.99   
6   3041230                      Windrose           29.99        29.99   
7   3110760                Alabaster Dawn           24.99        21.24   
9   3404260                 Dead as Disco           24.99        19.99   
0    311210  Call of Duty®: Black Ops III           59.99        19.79   
5   2582320                       Mixtape           19.99        17.99   
8   3124540                  Far Far West           19.99        17.99   
12  3672400                       Farever           19.99        17.99   
25  4337790                    Akuma Rise           19.99        17.99   

    desconto     categoria  
4        0.0   top_sellers  
1       25.0      specials  
6        0.0   top_sellers  
7       15.0      specials  
9       20.0      specials  
0       67.0      specials  
5       10.0      specials  
8       10.0      specials  
12      10.0      specials  
25      10.0  new_releases